# WSDP - Wi-Fi Sensing Data Processing

This notebook demonstrates the main features of the `wsdp` library:

1. Downloading datasets
2. Reading and processing CSI data
3. Running algorithms (denoising, phase calibration)
4. Visualization
5. Running the training pipeline
6. Inference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from wsdp import pipeline, download, predict, predict_single
from wsdp.algorithms import wavelet_denoise_csi, phase_calibration
from wsdp.algorithms.visualization import (
    plot_csi_heatmap,
    plot_denoising_comparison,
    plot_phase_calibration,
)
from wsdp.readers import list_datasets

print("Available datasets:", list_datasets())

## 1. Download a Dataset

```python
# Download with interactive auth
download('widar', '/data/widar')

# Download with JWT token (non-interactive)
download('elderAL', '/data/elder', token='your-jwt-token')

# Download with email/password (non-interactive)
download('elderAL', '/data/elder', email='user@example.com', password='secret')
```

## 2. Generate Sample CSI Data for Demo

In [ ]:
# Generate synthetic CSI data: T=200 time steps, F=30 subcarriers, A=3 antennas
T, F, A = 200, 30, 3
np.random.seed(42)

# Create CSI with phase slope and noise
subcarriers = np.arange(F)
csi_data = np.zeros((T, F, A), dtype=complex)

for t in range(T):
    for a in range(A):
        phase_slope = 0.5 * subcarriers + 0.2 * a
        amplitude = 1.0 + 0.3 * np.sin(0.1 * t) * np.ones(F)
        noise = 0.2 * np.random.randn(F) + 0.2j * np.random.randn(F)
        csi_data[t, :, a] = amplitude * np.exp(1j * phase_slope) + noise

print(f"CSI shape: {csi_data.shape} (T={T}, F={F}, A={A})")

## 3. Phase Calibration

In [ ]:
calibrated = phase_calibration(csi_data)

fig = plot_phase_calibration(csi_data, calibrated, antenna_idx=0, time_idx=0)
plt.show()

## 4. Wavelet Denoising

In [ ]:
denoised = wavelet_denoise_csi(csi_data)

fig = plot_denoising_comparison(csi_data, denoised, antenna_idx=0, subcarrier_idx=5)
plt.show()

## 5. CSI Heatmap Visualization

In [ ]:
fig = plot_csi_heatmap(csi_data, antenna_idx=0, title="CSI Amplitude Heatmap (Antenna 0)")
plt.show()

## 6. Full Training Pipeline

```python
from wsdp import pipeline

pipeline(
    input_path='/path/to/data',
    output_folder='/path/to/output',
    dataset='widar',
    learning_rate=1e-3,  # override default
    num_epochs=50,        # override default
)

# Or use YAML config:
pipeline(
    input_path='/path/to/data',
    output_folder='/path/to/output',
    dataset='widar',
    config_file='config.yaml',
)
```

## 7. Inference

```python
from wsdp import predict, predict_single

# Batch prediction
csi_batch = np.random.randn(10, 200, 30, 3) + 1j * np.random.randn(10, 200, 30, 3)
predictions = predict(csi_batch, 'best_checkpoint_42.pth', num_classes=6)
print(f"Predictions: {predictions}")

# Single sample
csi_single = np.random.randn(200, 30, 3) + 1j * np.random.randn(200, 30, 3)
pred = predict_single(csi_single, 'best_checkpoint_42.pth', num_classes=6)
print(f"Predicted class: {pred}")
```